# Exercise: the distillation loss for one example

**Goal:** compute the top-K self-distillation loss for one example from `runs/distill/examples.jsonl`, by hand, step by step.

- **Teacher:** Qwen3.5-9B with the hint in its context (`teacher_prompt`).
- **Student:** the same model without the hint (`student_prompt`).
- Both read the same **target**: the tokens the model actually wrote at the hint turn.

At every target position $i$, the teacher's top-K tokens, renormalized to sum to 1, give a distribution $p_T(\cdot \mid i)$ that is zero outside those K tokens. The loss pushes the student's **full** distribution $p_S(\cdot \mid i)$, over the whole vocabulary, toward it:

$$\mathcal{L} = \frac{1}{T}\sum_{i=1}^{T} \sum_{k \in \text{top-}K_T(i)} p_T(k \mid i)\,\big(\log p_T(k \mid i) - \log p_S(k \mid i)\big)$$

That's KL(teacher ‖ student), averaged over the $T$ target positions. Since $p_T$ is zero outside its top-K, only those K tokens appear in the sum, but $\log p_S$ is the student's unrenormalized log-prob: probability the student puts on other tokens counts against it. This is the loss training minimizes (step 5 computes it the way training will, with Tinker's `cross_entropy`).

**Steps**
1. Build the teacher and student sequences.
2. Get the teacher's top-K tokens and log-probs at the target positions (Tinker).
3. Get the student's log-probs for those same tokens: a Tinker training-client forward pass.
4. Compute the loss yourself.
5. Compute the same loss the Tinker way: the training datum and Tinker's `cross_entropy`, as training will.
6. *(Optional)* Look at where the teacher and student disagree most.

Each step has a **TODO** cell for you and a **check** cell after it. Run the check: it tells you whether your code is right.

**Cost:** the default example has an 11.7k-token context and 107 target tokens, so each Tinker call is about a cent.

In [131]:
# Setup (given). Run this first.
import json
import os
from pathlib import Path
import sys

import numpy as np
import tinker
import torch

# Jupyter runs from notebooks/; make the repo root the working directory and importable.
REPO = Path.cwd() if (Path.cwd() / "sd").exists() else Path.cwd().parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from sd.renderer import get_agent_renderer


# TINKER_API_KEY from the environment, or from the gitignored .env
if "TINKER_API_KEY" not in os.environ and (REPO / ".env").exists():
    for line in (REPO / ".env").read_text().splitlines():
        if line.strip() and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip())
assert "TINKER_API_KEY" in os.environ, "set TINKER_API_KEY or create .env"

MODEL = "Qwen/Qwen3.5-9B"
K = 20

examples = [json.loads(l) for l in open("runs/distill/examples.jsonl")]
example = next(e for e in examples if e["instance_id"] == "sphinx-doc__sphinx-8638")  # smallest context: cheap

renderer = get_agent_renderer(MODEL)
tok = renderer.tokenizer
service = tinker.ServiceClient()
sampling_client = service.create_sampling_client(base_model=MODEL)       # teacher (step 2)
training_client = service.create_lora_training_client(base_model=MODEL)  # student forward pass (step 3)

print(example["instance_id"], "| hint turn", example["hint_turn"])
print("student prompt", len(example["student_prompt"]), "tokens | teacher prompt", len(example["teacher_prompt"]),
      "| target", len(example["target"]))
print("hint:", example["hint"])
print("target starts:", repr(tok.decode(example["target"][:40])))

sphinx-doc__sphinx-8638 | hint turn 32
student prompt 11712 tokens | teacher prompt 11754 | target 107
hint: Note: you're close to your 40-turn limit, with about 9 turns left. You often keep exploring and forget to submit. Wrap up and call `task_complete`.
target starts: "Now I see the issue. There's a module-level variable `somevar` and a class attribute `foo.somvar` (instance variable). The automodule documentation might be auto-linking these"


## Step 1: build the sequences

The model reads the prompt followed by the target, so the full sequences are:

- `student_seq = student_prompt + target`
- `teacher_seq = teacher_prompt + target`

Compute `T` (the number of target tokens), `student_seq` and `teacher_seq`.

The target is the **last `T` tokens of both**, at different absolute positions, because the teacher's prompt is longer by the hint message (about 42 tokens). Because the target is at the end, the later steps can find its rows with `[-T:]` instead of tracking positions.

In [ ]:
# TODO
target = example["target"]
T = len(target)
student_seq = example['student_prompt'] + target
teacher_seq = example['teacher_prompt'] + target

In [ ]:
# Check step 1
assert T == len(example["target"])
assert student_seq == example["student_prompt"] + target and teacher_seq == example["teacher_prompt"] + target
assert student_seq[-T:] == target and teacher_seq[-T:] == target
print("step 1 OK")

## Step 2: the teacher's top-K at the target positions

Ask Tinker for top-K **prompt** log-probs over the whole `teacher_seq`. We don't need any generation, so `max_tokens=1`.

```python
response = sampling_client.sample(
    prompt=tinker.ModelInput.from_ints(teacher_seq),
    num_samples=1,
    sampling_params=tinker.SamplingParams(max_tokens=1),
    include_prompt_logprobs=True,   # required: without it, no top-K comes back
    topk_prompt_logprobs=K,
).result()
response.topk_prompt_logprobs_np.token_ids  # int32,   shape (len(teacher_seq), K)
response.topk_prompt_logprobs_np.logprobs   # float32, shape (len(teacher_seq), K)
```

**Indexing:** row `i` is the teacher's distribution for the token **at position `i`**, given the tokens before it. Row 0 is empty, since nothing comes before it. The target's rows are the last `T`: `[-T:]`.

Compute, as torch tensors of shape `(T, K)`:
- `teacher_topk_ids`: the teacher's top-K token IDs at each target position
- `teacher_topk_lp`: their log-probs

In [134]:
# TODO
response_t = sampling_client.sample(
    prompt=tinker.ModelInput.from_ints(teacher_seq),
    num_samples=1,
    sampling_params=tinker.SamplingParams(max_tokens=1),
    include_prompt_logprobs=True,
    topk_prompt_logprobs=K
).result()


In [135]:
teacher_topk_ids = response_t.topk_prompt_logprobs_np.token_ids[-T:]
teacher_topk_lp = response_t.topk_prompt_logprobs_np.logprobs[-T:]

In [136]:
# Check step 2
# Accept numpy arrays too (np.ascontiguousarray handles views like a[:, ::-1]).
_t = lambda x: torch.as_tensor(np.ascontiguousarray(x)) if isinstance(x, np.ndarray) else x
teacher_topk_ids, teacher_topk_lp = _t(teacher_topk_ids).long(), _t(teacher_topk_lp).float()
assert teacher_topk_ids.shape == (T, K) and teacher_topk_lp.shape == (T, K)
assert torch.all(teacher_topk_lp <= 0) and torch.all(teacher_topk_lp[:, :-1] >= teacher_topk_lp[:, 1:]), "rows should be sorted, best first"
in_topk = (teacher_topk_ids == torch.tensor(target).unsqueeze(1)).any(1).float().mean().item()
mass = teacher_topk_lp.exp().sum(1)
print(f"step 2 OK | the written token is in the teacher's top-{K} at {in_topk:.0%} of positions | "
      f"top-{K} covers {mass.mean():.1%} of the teacher's probability on average (min {mass.min():.1%})")

step 2 OK | the written token is in the teacher's top-20 at 100% of positions | top-20 covers 99.8% of the teacher's probability on average (min 95.1%)


## Step 3: the student's log-probs for the same tokens

We need the **student's** log-prob for each of the teacher's K tokens at each target position. The sampler can't do this here: it scores at most 20 distinct token IDs per request, and the teacher's top-20 over a whole target involve about 1,000.

The **training client's forward pass** can. It takes a `tinker.Datum`: the input tokens plus, per row, K `target_tokens` to score. It returns the student's log-prob of every one of them:

```python
datum = tinker.Datum(
    model_input=tinker.ModelInput.from_ints(student_seq[:-1]),       # the model predicts token i+1 from tokens <= i
    loss_fn_inputs={
        "target_tokens": tinker.TensorData.from_torch(target_tokens),  # int64, shape (len(student_seq) - 1, K)
        "weights": tinker.TensorData.from_torch(weights),              # float32, same shape
    },
)
out = training_client.forward([datum], loss_fn="cross_entropy").result()   # forward only: no weight update
out.loss_fn_outputs[0]["logprobs"].to_torch()   # shape (len(student_seq) - 1, K): the student's log-prob of each target cell
```

**Indexing:** **row `i` predicts the token at position `i + 1`.** So the target token at position `p` lives in row `p - 1`. (Step 2 was different: there, row `i` was position `i`.) There are `len(student_seq) - 1` rows, and the target's tokens are the last `T` positions, so their rows are again the last `T`: `[-T:]`.

Fill `target_tokens` with the teacher's top-K IDs on the target rows. Other rows can hold anything, e.g. zeros.

**`weights`:** Tinker's `cross_entropy` requires the key, but its values don't change the log-probs you read here, so use zeros. The weights only define the *loss* (`-weight * logprob` per cell); step 5 fills them in for real.

Compute `student_lp_on_teacher_topk`, shape `(T, K)`: the student's log-probs for `teacher_topk_ids`.

In [137]:
# TODO
target_tokens = torch.concat([torch.zeros(len(student_seq) - T - 1, K, dtype=int), teacher_topk_ids])
weights = torch.zeros(len(student_seq) - 1, K)  # required by cross_entropy; values don't affect the log-probs
datum = tinker.Datum(
    model_input=tinker.ModelInput.from_ints(student_seq[:-1]),
    loss_fn_inputs={
        "target_tokens": target_tokens,
        "weights": weights
    }

)
out = training_client.forward([datum], loss_fn="cross_entropy").result()

In [138]:
student_lp_on_teacher_topk = out.loss_fn_outputs[0]['logprobs'].to_torch()[-T:]

In [139]:
# Check step 3
# Accept numpy arrays too (np.ascontiguousarray handles views like a[:, ::-1]).
_t = lambda x: torch.as_tensor(np.ascontiguousarray(x)) if isinstance(x, np.ndarray) else x
student_lp_on_teacher_topk = _t(student_lp_on_teacher_topk).float()
assert student_lp_on_teacher_topk.shape == (T, K) and torch.all(student_lp_on_teacher_topk <= 0)
# The student's log-prob of the token it actually wrote should match what vLLM recorded during the rollout.
match = teacher_topk_ids == torch.tensor(target).unsqueeze(1)              # where the written token sits in each row
has = match.any(1)
written_lp = student_lp_on_teacher_topk[has, match[has].float().argmax(1)]
diff = (written_lp - torch.tensor(example["target_logprobs"])[has]).abs()
print(f"step 3 OK | student vs recorded log-prob of the written token ({has.sum()}/{T} positions): "
      f"median |diff| {diff.median():.4f}, max {diff.max():.3f}")
assert diff.median() < 0.05, "student log-probs don't match the rollout: check the row/position offset"

step 3 OK | student vs recorded log-prob of the written token (107/107 positions): median |diff| 0.0000, max 0.111


## Step 4: the loss

Implement `topk_kl(teacher_lp, student_lp)`. Both inputs have shape `(T, K)` and hold log-probs over the **same** K tokens per row: the teacher's top-K. It returns a tensor of shape `(T,)`: the KL at each position.

1. Renormalize the **teacher** over its K tokens: `log p_T = log_softmax(teacher_lp)` per row, and `p_T = log p_T.exp()`. The top-K covers most but not all of the teacher's probability; renormalizing makes $p_T$ a proper distribution. Use the renormalized $\log p_T$ in the formula too, not the raw `teacher_lp`, or the KL can go negative.
2. Leave the **student** as it is: `student_lp` holds its full-vocabulary log-probs, which is what it samples from. Don't renormalize it over the K tokens.
3. Per row: $\sum_k p_T(k)\,(\log p_T(k) - \log p_S(k))$.

Call it on `teacher_topk_lp` and `student_lp_on_teacher_topk`. Then `loss = per_position.mean()`, the average over target positions.

**Why not renormalize the student too?** That gives a different KL, over the K tokens only. With $s$ = the student's total probability on the teacher's K tokens:

$$\text{KL}_\text{full} = \text{KL}_\text{shape} + (-\ln s)$$

$\text{KL}_\text{shape}$ only compares how the two rank the teacher's K tokens. It can be 0 while the student puts 90% of its probability elsewhere, and the student samples from its full distribution, so it would rarely write what the teacher wants. The $-\ln s$ term catches that.

In [141]:
# TODO
def topk_kl(teacher_lp: torch.Tensor, student_lp: torch.Tensor) -> torch.Tensor:
    # renormalize teacher_lp
    teacher_lp = torch.log_softmax(teacher_lp, 1)  # (T, K)
    p_T = torch.exp(teacher_lp) # (T, K)
    # per row, aka, per position
    return (p_T * (teacher_lp - student_lp)).sum(1)

per_position = topk_kl(teacher_topk_lp, student_lp_on_teacher_topk)
loss = per_position.mean()

In [142]:
# Check step 4
# Accept numpy arrays too (np.ascontiguousarray handles views like a[:, ::-1]).
_t = lambda x: torch.as_tensor(np.ascontiguousarray(x)) if isinstance(x, np.ndarray) else x
per_position = _t(per_position).float()
t = torch.log(torch.tensor([[0.5, 0.3, 0.2]])); s = torch.log(torch.tensor([[0.2, 0.3, 0.5]]))
assert torch.allclose(topk_kl(t, s), torch.tensor([0.274887]), atol=1e-5), topk_kl(t, s)          # worked out by hand
assert torch.allclose(topk_kl(t, t), torch.zeros(1), atol=1e-6)                                    # identical -> 0
assert torch.allclose(topk_kl(t + np.log(0.9), s), topk_kl(t, s), atol=1e-5), \
    "the teacher's top-K covering only 90% shouldn't matter: renormalize the teacher (log p_T and p_T)"
assert torch.allclose(topk_kl(t, s + np.log(0.5)), topk_kl(t, s) + np.log(2), atol=1e-5), \
    "the student putting half its probability elsewhere should cost -ln 0.5 = 0.693: don't renormalize the student"
assert torch.allclose(topk_kl(t, t + np.log(0.1)), torch.tensor([2.302585]), atol=1e-5)          # same shape, s = 0.1 -> -ln 0.1
assert per_position.shape == (T,) and torch.all(per_position >= -1e-6)
print(f"step 4 OK | loss {loss.item():.4f} | per position: median {per_position.median():.4f}, max {per_position.max():.3f}")

step 4 OK | loss 0.0213 | per position: median 0.0007, max 0.859


## Step 5: the same loss, the Tinker way

In step 4 you computed the KL yourself from log-probs. Training doesn't do that. It hands Tinker a **training datum** and calls `training_client.forward_backward([datum], loss_fn="cross_entropy")`, Tinker's built-in loss: per cell, `-weight * logprob`, summed. Here you compute the same loss that way, and check that it's your step-4 loss.

Build the training datum: step 3's datum, but with real weights.

- `train_weights`: the teacher's top-K **probabilities**, renormalized to sum to 1 per row (that's $p_T$ from step 4), on the target rows, and `0` elsewhere. Shape `(len(student_seq) - 1, K)`.
- `train_datum`: step 3's datum with `train_weights` as `"weights"`.
- `train_out = training_client.forward([train_datum], loss_fn="cross_entropy").result()`: a forward pass only, so it doesn't train. Training would call `forward_backward` on exactly this datum.

`train_out` gives you:
- `train_out.loss_fn_outputs[0]["elementwise_loss"]`, shape `(len(student_seq) - 1, K)`: per cell, `-weight * logprob`. Rows with weight 0 are 0.
- `train_out.metrics["loss:sum"]`: the sum of all cells. A **sum** over positions, not a mean.

That's cross-entropy with soft targets: the weights are $p_T$, and the log-probs are the student's full-vocabulary ones. Per position:

$$\text{CE} = -\sum_k p_T(k)\log p_S(k) = \underbrace{-\sum_k p_T(k)\log p_T(k)}_{H(p_T),\ \text{the teacher's entropy}} + \text{KL}_\text{full}$$

The teacher is frozen, so $H(p_T)$ is a constant: it has no gradient, and minimizing CE is minimizing your step-4 KL. It does mean the CE never reaches 0; its floor is $H$.

Compute:
- `ce_per_pos`, shape `(T,)`: CE per target position, from Tinker's `elementwise_loss`
- `ce_sum`: CE summed over target positions, **recomputed by you** from `student_lp_on_teacher_topk` and $p_T$ (a number)
- `H`, shape `(T,)`: the teacher's entropy per position

Questions to answer:
- How big is $H$ next to the KL? What would the training loss curve look like if the student matched the teacher perfectly?
- `loss:sum` grows with $T$. In a batch with a 107-token and a 2,000-token target, which example drives the gradient? Is that what we want?

For reference, the cookbook builds the same datum in `_collect_topk_for_datum` in `.venv/lib/python3.12/site-packages/tinker_cookbook/distillation/train_off_policy.py`.

In [ ]:
# TODO

In [ ]:
# Check step 5
_t = lambda x: torch.as_tensor(np.ascontiguousarray(x)) if isinstance(x, np.ndarray) else x
train_weights = _t(train_weights).float()
ce_per_pos, H = _t(ce_per_pos).float(), _t(H).float()
ce_sum = float(ce_sum)
assert train_weights.shape == (len(student_seq) - 1, K)
assert torch.all(train_weights[:-T] == 0) and torch.allclose(train_weights[-T:].sum(1), torch.ones(T), atol=1e-4), \
    "weights: p_T on the target rows (each summing to 1), 0 elsewhere"
assert ce_per_pos.shape == (T,) and H.shape == (T,)
assert abs(ce_sum - train_out.metrics["loss:sum"]) < 1e-2 * max(1.0, abs(ce_sum)), (ce_sum, train_out.metrics["loss:sum"])
assert torch.allclose(ce_per_pos.sum(), torch.tensor(ce_sum), rtol=1e-3)
assert torch.allclose(ce_per_pos, H + per_position, atol=1e-3), "CE should be H + your step-4 KL at every position"
print(f"step 5 OK | loss:sum {train_out.metrics['loss:sum']:.3f} | per position, mean: "
      f"CE {ce_per_pos.mean():.4f} = H {H.mean():.4f} + KL {per_position.mean():.4f}")

## Step 6 (optional): where do the teacher and student disagree?

First split your step-4 KL into its two parts, as tensors of shape `(T,)`. With $s$ = the student's total probability on the teacher's top-K at each position:
- `kl_shape`: $\text{KL}_\text{shape}$, the KL with the student renormalized over the K tokens (how the two rank the teacher's tokens)
- `missing`: $-\ln s$ (how much probability the student puts on tokens the teacher doesn't consider)

They should add up to `per_position`.

Then find the **10 positions with the highest KL**. For each, print one line with:
- the position within the target, and the written token (decode it with `tok.decode([id])`)
- the KL, and its two parts
- the teacher's top 3 tokens with their probabilities (renormalized), next to the student's probabilities for those same tokens
- `s`

For example:

```
pos   0 | KL 0.853 (shape 0.xx + missing 0.xx) | wrote 'Now' | teacher: 'Let' 0.78, 'Now' 0.12, 'I' 0.07 | student: 'Let' 0.xx, 'Now' 0.xx, 'I' 0.xx | s 0.xx
```

Questions to answer:
- Are the big disagreements at the start of the turn, for example its first thinking tokens, or spread out?
- Does the teacher's preferred alternative look like wrapping up (*the fix works*, *review the diff*, *submit*) while the written token keeps exploring (*also*, *test*, *check*)?
- Which part dominates the KL, overall and at the top 10: **ranking** (`kl_shape`) or **missing mass** (`missing`)?
- *Bonus:* plot `per_position` against position within the target.

In [ ]:
# TODO
values, indices = per_position.topk(10)

for i in range(10):
    idx = indices[i]
    target_id = target[idx]
    print(target_id)
    target_tok = tok.decode(target_id)
    print(target_tok)
    print(teacher_topk_lp[idx, :3])
    print(tok.decode(teacher_topk_ids[idx, :3]))
    print()


6820
Now
tensor([-0.2462, -2.1212, -2.6212])
LetNowI

1436
 see
tensor([-0.3706, -1.7456, -2.4956])
 understand need have

2532
 There
tensor([-0.6207, -1.6207, -1.8707])
 The In There

4845
 autom
tensor([-1.1274, -1.7524, -1.8774])
 issue autod problem

1380
 try
tensor([-0.6468, -1.8968, -2.3968])
 build try look

353
 I
tensor([-0.5276, -0.9026, -5.5276])
 let I,

198


tensor([-0.2247, -1.7247, -4.0997])



 But

13
.
tensor([-0.5311, -1.7811, -2.4061])
.! clearly

279
 the
tensor([-0.0287, -5.0287, -5.4037])
 the -.

4125
 issue
tensor([-0.1733, -3.0483, -3.2983])
 issue problem example



In [ ]:
# Check step 6
_t = lambda x: torch.as_tensor(np.ascontiguousarray(x)) if isinstance(x, np.ndarray) else x
kl_shape, missing = _t(kl_shape).float(), _t(missing).float()
assert kl_shape.shape == (T,) and missing.shape == (T,)
assert torch.allclose(per_position, kl_shape + missing, atol=1e-3), "KL_full should be KL_shape + (-ln s)"
assert torch.all(kl_shape >= -1e-6) and torch.all(missing >= -1e-6)
print(f"step 6 OK | KL per position, mean: {per_position.mean():.4f} = shape {kl_shape.mean():.4f} + missing {missing.mean():.4f}")